# CelebA Generation with Embedded Interpolants

Replicating the **CelebA 128×128** experiment from
*Coeurdoux et al. 2026, Figure 4* (arXiv:2602.20070).

Their setup: ensemble of 25 weak U-Nets (5 epochs each), kernelized via stochastic interpolants → coherent faces.
Our setup: same target distribution, but our method (BW-OT in RKHS, no neural network) instead of the
weak-pretrained-models route.

## Pipeline

`128×128 RGB → flatten 49152 → PCA(d_pca) → EmbeddedInterpolants → inverse PCA → 128×128 RGB grid`

This is the same pipeline as faces (LFW) at higher resolution and in colour.

**Note: this notebook is provided as a scaffold.**  CelebA is large (~1.4 GB) and training-time
EmbeddedInterpolants on color images of dim 49152 (or even d_pca=256) is non-trivial — expect
order-of-minutes per iteration. Run on a workstation, not laptop.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path

from src import EmbeddedInterpolants

np.random.seed(0)
plt.rcParams['figure.dpi'] = 110


## Loading CelebA

Two options:

### A) `torchvision`
```bash
pip install torchvision
```
Then run the cell below.  First run downloads ~1.4 GB; afterwards loads from cache.

### B) Manual download
1. Get `img_align_celeba.zip` from the official Google Drive link
2. Unzip into `~/data/celeba/img_align_celeba/`
3. Use the alternative loader (commented out below).


In [ ]:
# ── option A: torchvision (uncomment to use) ────────────────────────────
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader
#
# tfm = transforms.Compose([
#     transforms.CenterCrop(178),       # CelebA aligned crop
#     transforms.Resize(128),
#     transforms.ToTensor(),            # [3, 128, 128] in [0, 1]
# ])
# ds = datasets.CelebA(root='~/data', split='train', target_type=[], download=True, transform=tfm)
#
# # take a subset (full CelebA is 162k images, way too many for our purposes)
# N_USE = 5000
# loader = DataLoader(ds, batch_size=64, shuffle=True, num_workers=2)
# imgs = []
# for batch, _ in loader:
#     imgs.append(batch.numpy())
#     if sum(b.shape[0] for b in imgs) >= N_USE:
#         break
# imgs = np.concatenate(imgs, axis=0)[:N_USE]   # (N, 3, 128, 128) in [0, 1]
# imgs = imgs.transpose(0, 2, 3, 1)             # (N, 128, 128, 3)
# print(imgs.shape, imgs.dtype, imgs.min(), imgs.max())


# ── option B: manual folder of jpg ─────────────────────────────────────
# from PIL import Image
# def load_celeba_folder(folder, n=5000, size=128):
#     paths = sorted(Path(folder).glob('*.jpg'))[:n]
#     out = np.empty((len(paths), size, size, 3), dtype=np.float32)
#     for i, p in enumerate(paths):
#         im = Image.open(p).convert('RGB')
#         w, h = im.size
#         left = (w - 178) // 2; top = (h - 178) // 2     # CelebA aligned crop ~178
#         im = im.crop((left, top, left + 178, top + 178)).resize((size, size), Image.LANCZOS)
#         out[i] = np.asarray(im, dtype=np.float32) / 255.0
#     return out
#
# imgs = load_celeba_folder('~/data/celeba/img_align_celeba/', n=5000)


# ── placeholder (so cell runs without download) ────────────────────────
# Replace with one of the loaders above before training.
imgs = None
print('imgs not loaded — uncomment one of the loaders above')


## Train EmbeddedInterpolants on CelebA

This is the heavy cell.  Default `d_pca=128, n_iterations=8, K_steps=60` — adjust if memory tight.


In [ ]:
def run_celeba(imgs, d_pca=128, n_iterations=8, K_steps=60,
                noise_level=1.5, q=0.5, q_final=0.15,
                gamma=1e-2, gamma_final=1e-7, n_inducing=150, seed=0):
    rng = np.random.default_rng(seed)
    n, H, W, C = imgs.shape
    print(f'CelebA: {n} images, {H}×{W}×{C}')

    # train/test split
    perm = rng.permutation(n)
    n_train = int(0.9 * n)
    train_idx, test_idx = perm[:n_train], perm[n_train:]
    X_train = imgs[train_idx].reshape(n_train, -1)
    X_test  = imgs[test_idx].reshape(n - n_train, -1)

    # standardise to (approx) zero mean, unit variance per pixel
    mu  = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True) + 1e-6
    X_train_s = (X_train - mu) / std
    X_test_s  = (X_test  - mu) / std

    # PCA
    pca = PCA(n_components=d_pca, whiten=False).fit(X_train_s)
    Z_train = pca.transform(X_train_s).astype(np.float32)
    print(f'PCA: d_pca={d_pca},  explained variance={pca.explained_variance_ratio_.sum():.3f}')

    sigma_z = Z_train.std(axis=0, keepdims=True) + 1e-8
    Z_train_n = Z_train / sigma_z

    # fit
    model = EmbeddedInterpolants(
        sigma_k=None, bandwidth_method='quantile',
        q=q, q_final=q_final, gamma=gamma, gamma_final=gamma_final,
        K_steps=K_steps, n_inducing=n_inducing,
        noise_level=noise_level, rescale=True,
    )
    src = rng.standard_normal((n_train, d_pca)).astype(np.float32)
    model.fit(src, Z_train_n, n_iterations=n_iterations, verbose=True)

    # generate
    n_gen = 16
    src_new = rng.standard_normal((n_gen, d_pca)).astype(np.float32)
    res = model.transport(src_new, verbose=False)
    Z_gen = res['particles'] * sigma_z
    X_gen_s = pca.inverse_transform(Z_gen)
    X_gen = X_gen_s * std + mu
    X_gen = np.clip(X_gen, 0.0, 1.0).reshape(n_gen, H, W, C)

    # show: 2 rows of real (8) / 2 rows of generated (8)
    fig, axes = plt.subplots(4, 8, figsize=(14, 7.2))
    real_show = imgs[test_idx[:16]]
    for j in range(8):
        axes[0, j].imshow(real_show[j]);     axes[0, j].axis('off')
        axes[1, j].imshow(real_show[8 + j]); axes[1, j].axis('off')
        axes[2, j].imshow(X_gen[j]);         axes[2, j].axis('off')
        axes[3, j].imshow(X_gen[8 + j]);     axes[3, j].axis('off')
    axes[0, 0].set_title('ground truth (top 2 rows)', loc='left', fontsize=12)
    axes[2, 0].set_title('generated (bottom 2 rows)', loc='left', fontsize=12)
    plt.tight_layout()
    plt.savefig('celeba_grid.png', dpi=150, bbox_inches='tight')
    plt.show()

    return {'model': model, 'pca': pca, 'mu': mu, 'std': std, 'sigma_z': sigma_z,
            'X_gen': X_gen}


# uncomment when imgs is loaded:
# result = run_celeba(imgs, d_pca=128, n_iterations=8)


## Tuning notes

- **`d_pca`**: 128 is a reasonable starting point.  CelebA at 128×128×3 has ~50k pixels;
  PCA captures ~85% variance at d=128 and ~95% at d=300.  The lifting ratio η starts breaking down
  above d~250 in our experience.
- **`noise_level`**: 1.5 works for most images; if generations look "too noisy" reduce to 0.8.
- **`n_iterations`**: 8 typically sufficient; on CelebA you may benefit from 12–15
  given the larger and more diverse target distribution.
- **`n_inducing`**: 150 is fine; raising to 250 helps quality but quadratic cost in memory.
- **Memory**: the Nyström M×M factors and the BW solve are O(M²) per iteration. With M=150 and
  n_train=5000, expect ~2–4 minutes per iteration on a single CPU.

## Comparison with Coeurdoux et al. Figure 4

The paper shows two rows: top = single weak U-Net (5 epochs), bottom = ensemble of 25 weak models
combined via their kernel system. Our method is conceptually different (no pretrained models;
just BW-OT in RKHS with kernel feature map) but should yield qualitatively similar coherent faces
at this resolution, modulo PCA bottleneck.
